# Financial News Sentiment Analysis
## Intro
This notebook implements a sentiment analysis pipeline for financial news data. The project aims to analyze sentiment in financial news articles from two major sources:
- [Bloomberg Financial News Dataset](https://huggingface.co/datasets/danidanou/Bloomberg_Financial_News)
- [Reuters Financial News Dataset](https://huggingface.co/datasets/danidanou/Reuters_Financial_News?library=pandas)
## Methodology
We will employ two state-of-the-art sentiment analysis models:
1. [FinBERT](https://huggingface.co/ProsusAI/finbert) - A specialized BERT model fine-tuned for financial sentiment analysis
2. [Google's Gemini](https://aistudio.google.com/prompts/new_chat) - A large language model for advanced text understanding

This approach allows us to compare and validate sentiment scores across different model architectures.

# Download dataset
Take about 1 minute to download

In [1]:
import pandas as pd

reuter_df = pd.read_parquet("hf://datasets/danidanou/Reuters_Financial_News/summ_financial_data.parquet.gzip")
bloomberg_df = pd.read_parquet("hf://datasets/danidanou/Bloomberg_Financial_News/bloomberg_financial_data.parquet.gzip")

In [2]:
# Print the first 5 rows of the dataframes
print("Reuters Dataframe:")
display(reuter_df.head())
print("Bloomberg Dataframe:")
display(bloomberg_df.head())

# print the shape of the dataframes
print("Reuters Dataframe Shape:")
print(reuter_df.shape)
print("Bloomberg Dataframe Shape:")
print(bloomberg_df.shape)



Reuters Dataframe:


,Headline,Journalists,Date,Link,Summary,Article
0,"Hitachi, GE boost alliance in nuclear power bu...",[],"Mon Nov 13, 2006 3:16am EST",http://www.reuters.com/article/2006/11/13/us-e...,TOKYO - Hitachi Ltd. ( 6501.T ) said on Monda...,The move comes a month after France's Areva C...
1,"Volvo to cut 1,000 staff at Virginia plant",[],"Mon Nov 13, 2006 8:45am EST",http://www.reuters.com/article/2006/11/13/us-a...,STOCKHOLM - Truck maker Volvo said on Monday ...,After years of strong demand truck makers see...
2,European banks hiding full pension obligations,[Andrew Hurst],"Mon Nov 13, 2006 3:15am EST",http://www.reuters.com/article/2006/11/13/us-f...,"ZURICH, Nov 13 (Reuter) - West European banks...",Since adopting International Financial Report...
3,"Hitachi, GE to form joint nuclear power ventures",[Mayumi Negishi],"Mon Nov 13, 2006 7:13am EST",http://www.reuters.com/article/2006/11/13/us-e...,TOKYO - Japan's Hitachi Ltd. and U.S. group G...,"The partnership would help Hitachi, Japan's b..."
4,Eddie Bauer agrees to be bought for $286 million,[],"Mon Nov 13, 2006 7:29am EST",http://www.reuters.com/article/2006/11/13/us-r...,- Eddie Bauer Holdings Inc. EBHI.O said and i...,The cash deal is expected to provide Eddie Ba...


Bloomberg Dataframe:


,Headline,Journalists,Date,Link,Article
0,"Ivory Coast Keeps Cocoa Export Tax Below 22%, ...",[Baudelaire Mieu],2011-10-06 15:14:20,http://www.bloomberg.com/news/2011-10-06/ivory...,"Export taxes on cocoa beans from Ivory Coast ,..."
1,USDA Boxed Beef Cutout Closing Prices for Octo...,[Michael Carone],2011-10-06 20:22:42,http://www.bloomberg.com/news/2011-10-06/usda-...,October 6 (Bloomberg) -- This table details bo...
2,U.S. September Small Business Jobs Summary,[Alex Tanzi],2011-10-06 19:00:00,http://www.bloomberg.com/news/2011-10-06/u-s-s...,U.S. small business plans to hire declined in ...
3,Greece’s GSEE Says Won’t Meet For Talks With T...,[Natalie Weeks],2011-10-06 14:45:34,http://www.bloomberg.com/news/2011-10-06/greec...,"Greece ’s biggest private sector union group, ..."
4,Clean-Tech Companies Should Get 10-Year Tax Br...,[Ari Levy],2011-10-06 18:34:41,http://www.bloomberg.com/news/2011-10-06/clean...,"Reed Hundt, head of the Coalition for Green Ca..."


Reuters Dataframe Shape:
(105359, 6)
Bloomberg Dataframe Shape:
(446762, 5)


# Data Cleaning

## Lowercase the column names

In [3]:
reuter_df.columns = reuter_df.columns.str.lower()
bloomberg_df.columns = bloomberg_df.columns.str.lower()

## Remove columns

In [4]:
# Lowercase the column names
columns_to_keep = ['date', 'headline', 'article']
reuter_df_cleaned = reuter_df[columns_to_keep]
bloomberg_df_cleaned = bloomberg_df[columns_to_keep]

In [5]:
print("Reuters Dataframe:")
display(reuter_df_cleaned.head())
print("Bloomberg Dataframe:")
display(bloomberg_df_cleaned.head())

Reuters Dataframe:


,date,headline,article
0,"Mon Nov 13, 2006 3:16am EST","Hitachi, GE boost alliance in nuclear power bu...",The move comes a month after France's Areva C...
1,"Mon Nov 13, 2006 8:45am EST","Volvo to cut 1,000 staff at Virginia plant",After years of strong demand truck makers see...
2,"Mon Nov 13, 2006 3:15am EST",European banks hiding full pension obligations,Since adopting International Financial Report...
3,"Mon Nov 13, 2006 7:13am EST","Hitachi, GE to form joint nuclear power ventures","The partnership would help Hitachi, Japan's b..."
4,"Mon Nov 13, 2006 7:29am EST",Eddie Bauer agrees to be bought for $286 million,The cash deal is expected to provide Eddie Ba...


Bloomberg Dataframe:


,date,headline,article
0,2011-10-06 15:14:20,"Ivory Coast Keeps Cocoa Export Tax Below 22%, ...","Export taxes on cocoa beans from Ivory Coast ,..."
1,2011-10-06 20:22:42,USDA Boxed Beef Cutout Closing Prices for Octo...,October 6 (Bloomberg) -- This table details bo...
2,2011-10-06 19:00:00,U.S. September Small Business Jobs Summary,U.S. small business plans to hire declined in ...
3,2011-10-06 14:45:34,Greece’s GSEE Says Won’t Meet For Talks With T...,"Greece ’s biggest private sector union group, ..."
4,2011-10-06 18:34:41,Clean-Tech Companies Should Get 10-Year Tax Br...,"Reed Hundt, head of the Coalition for Green Ca..."


## Convert date to ISO8601

In [6]:
def convert_datetime_column(df: pd.DataFrame, column_name: str, target_tz: str = 'US/Eastern') -> pd.DataFrame:
    """
    Standardizes and converts a datetime column to ISO 8601 format with specified timezone.
    Handles both timezone-naive and timezone-aware data.

    Args:
        df (pd.DataFrame): The pandas dataframe containing the column to convert.
        column_name (str): The name of the column to convert.
        target_tz (str): Target timezone to convert to. Defaults to 'US/Eastern'.

    Returns:
        pd.DataFrame: The original dataframe with the specified column converted to ISO 8601 format.
    """
    # Create a copy to avoid modifying the original
    df = df.copy()
    
    # Convert to datetime if not already
    df[column_name] = pd.to_datetime(df[column_name], errors='coerce')
    
    # Handle timezone conversion
    if df[column_name].dt.tz is None:
        # If timezone-naive, localize to target timezone
        df[column_name] = df[column_name].dt.tz_localize(target_tz, ambiguous='NaT', nonexistent='NaT')
    else:
        # If already timezone-aware, convert to target timezone
        df[column_name] = df[column_name].dt.tz_convert(target_tz)
    
    # Drop any rows with NaT (Not a Time) values due to DST transitions
    df = df.dropna(subset=[column_name])
    
    # Convert to ISO 8601 format
    df[column_name] = pd.to_datetime(df[column_name], format='ISO8601', errors='coerce')
    return df

In [7]:
reuter_df_cleaned = convert_datetime_column(reuter_df_cleaned, 'date')
bloomberg_df_cleaned = convert_datetime_column(bloomberg_df_cleaned, 'date')

print("Reuters Dataframe:")
display(reuter_df_cleaned.head())
print("Bloomberg Dataframe:")
display(bloomberg_df_cleaned.head())

Reuters Dataframe:


/var/folders/s2/gqpw_6dd5n7b8c1hzy_q5s0w0000gn/T/ipykernel_27487/148566061.py:18: FutureWarning: Parsed string "Mon Nov 13, 2006 3:16am EST" included an un-recognized timezone "EST". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  df[column_name] = pd.to_datetime(df[column_name], errors='coerce')


,date,headline,article
0,2006-11-13 03:16:00-05:00,"Hitachi, GE boost alliance in nuclear power bu...",The move comes a month after France's Areva C...
1,2006-11-13 08:45:00-05:00,"Volvo to cut 1,000 staff at Virginia plant",After years of strong demand truck makers see...
2,2006-11-13 03:15:00-05:00,European banks hiding full pension obligations,Since adopting International Financial Report...
3,2006-11-13 07:13:00-05:00,"Hitachi, GE to form joint nuclear power ventures","The partnership would help Hitachi, Japan's b..."
4,2006-11-13 07:29:00-05:00,Eddie Bauer agrees to be bought for $286 million,The cash deal is expected to provide Eddie Ba...


Bloomberg Dataframe:


,date,headline,article
0,2011-10-06 15:14:20-04:00,"Ivory Coast Keeps Cocoa Export Tax Below 22%, ...","Export taxes on cocoa beans from Ivory Coast ,..."
1,2011-10-06 20:22:42-04:00,USDA Boxed Beef Cutout Closing Prices for Octo...,October 6 (Bloomberg) -- This table details bo...
2,2011-10-06 19:00:00-04:00,U.S. September Small Business Jobs Summary,U.S. small business plans to hire declined in ...
3,2011-10-06 14:45:34-04:00,Greece’s GSEE Says Won’t Meet For Talks With T...,"Greece ’s biggest private sector union group, ..."
4,2011-10-06 18:34:41-04:00,Clean-Tech Companies Should Get 10-Year Tax Br...,"Reed Hundt, head of the Coalition for Green Ca..."


## Conbine two dataset

In [8]:
conbined_df = pd.concat([reuter_df_cleaned, bloomberg_df_cleaned])
print("Combined Dataframe:")
display(conbined_df.head())
display(conbined_df.shape)


Combined Dataframe:


,date,headline,article
0,2006-11-13 03:16:00-05:00,"Hitachi, GE boost alliance in nuclear power bu...",The move comes a month after France's Areva C...
1,2006-11-13 08:45:00-05:00,"Volvo to cut 1,000 staff at Virginia plant",After years of strong demand truck makers see...
2,2006-11-13 03:15:00-05:00,European banks hiding full pension obligations,Since adopting International Financial Report...
3,2006-11-13 07:13:00-05:00,"Hitachi, GE to form joint nuclear power ventures","The partnership would help Hitachi, Japan's b..."
4,2006-11-13 07:29:00-05:00,Eddie Bauer agrees to be bought for $286 million,The cash deal is expected to provide Eddie Ba...


(461427, 3)

## Drop duplicated and missing value

In [9]:
# Make stat of the combined dataframe
old_length = conbined_df.shape[0]

# Drop duplicatoin by headline
conbined_df = conbined_df.drop_duplicates(subset=['headline'])

# Drop missing value
conbined_df = conbined_df.dropna(subset=['date', 'headline'])
print("Combined Dataframe:")
display(conbined_df.head())
display(conbined_df.shape)

# Print the number of rows removed
print(f"Number of rows removed: {old_length - conbined_df.shape[0]}")


Combined Dataframe:


,date,headline,article
0,2006-11-13 03:16:00-05:00,"Hitachi, GE boost alliance in nuclear power bu...",The move comes a month after France's Areva C...
1,2006-11-13 08:45:00-05:00,"Volvo to cut 1,000 staff at Virginia plant",After years of strong demand truck makers see...
2,2006-11-13 03:15:00-05:00,European banks hiding full pension obligations,Since adopting International Financial Report...
3,2006-11-13 07:13:00-05:00,"Hitachi, GE to form joint nuclear power ventures","The partnership would help Hitachi, Japan's b..."
4,2006-11-13 07:29:00-05:00,Eddie Bauer agrees to be bought for $286 million,The cash deal is expected to provide Eddie Ba...


(451165, 3)

Number of rows removed: 10262


In [10]:
# Load the dataframe with sentiment scores but no articles
df_without_article = pd.read_csv('finbert_sentiment_inserted.csv')

# Merge the two dataframes on headline, keeping all rows from both (outer join)
# This will preserve both sentiment scores and articles where available
new_df = pd.merge(
    conbined_df,
    df_without_article,
    on='headline',
    how='outer',
    suffixes=('', '_sentiment')
)

# Drop any duplicate columns that might have been created during the merge
new_df = new_df.loc[:, ~new_df.columns.str.endswith('_sentiment')]

# Save the combined dataframe
new_df.to_csv('full.csv', index=False)

# Data Transformation

In [12]:
new_df.head()

# drop the missing value in headline
new_df = new_df.dropna(subset=['headline'])
new_df.to_csv('done.csv', index=False)



# Data Transformation

##  FinBert: Sentiment Score 

In [10]:
from transformers import pipeline
import pandas as pd
from tqdm import tqdm  # Import tqdm for progress bars
import numpy as np

# Initialize the sentiment analysis pipeline with FinBERT
pipe = pipeline("text-classification", model="prosusai/finbert", top_k=None)

def sentiment_to_score(positive: float, neutral: float, negative: float) -> float:
    """
    Convert sentiment scores to a single value between -1 and 1 using softmax.
    
    Args:
        positive (float): Positive sentiment score
        neutral (float): Neutral sentiment score
        negative (float): Negative sentiment score
        
    Returns:
        float: A single sentiment score between -1 and 1
    """
    # Apply softmax to get probabilities
    scores = np.array([positive, neutral, negative])
    exp_scores = np.exp(scores - np.max(scores))  # Subtract max for numerical stability
    probabilities = exp_scores / exp_scores.sum()
    
    # Convert to -1 to 1 scale
    # We weight positive as 1, neutral as 0, and negative as -1
    weighted_score = probabilities[0] * 1 + probabilities[1] * 0 + probabilities[2] * -1
    
    return weighted_score

def add_sentiment_scores(df: pd.DataFrame, column_name: str) -> pd.DataFrame:
    """
    Add sentiment scores to the DataFrame using FinBERT model.
    
    Args:
        df (pd.DataFrame): Input DataFrame containing text data
        column_name (str): Name of the column containing text to analyze
        
    Returns:
        pd.DataFrame: DataFrame with added sentiment scores:
            - sentiment_score_finbert: Combined score between -1 and 1
            - sentiment_positive_finbert: Raw positive sentiment score
            - sentiment_neutral_finbert: Raw neutral sentiment score
            - sentiment_negative_finbert: Raw negative sentiment score
    """
    print("Starting sentiment analysis...")
    
    # Create empty lists to hold the sentiment scores
    sentiment_scores = []
    positive_scores = []
    neutral_scores = []
    negative_scores = []
    
    # Create a progress bar for all texts
    with tqdm(total=len(df), desc="Processing texts", unit="text") as pbar:
        for text in df[column_name]:
            # Run sentiment analysis
            result = pipe(text)
            
            # Extract scores for positive, neutral, and negative sentiments
            positive_score = next(item['score'] for item in result[0] if item['label'] == 'positive')
            neutral_score = next(item['score'] for item in result[0] if item['label'] == 'neutral')
            negative_score = next(item['score'] for item in result[0] if item['label'] == 'negative')
            
            # Store individual scores
            positive_scores.append(positive_score)
            neutral_scores.append(neutral_score)
            negative_scores.append(negative_score)
            
            # Calculate and append the combined sentiment score
            sentiment_score = sentiment_to_score(positive_score, neutral_score, negative_score)
            sentiment_scores.append(sentiment_score)
            
            # Update progress bar
            pbar.update(1)
    
    # Add all scores to the DataFrame
    df['sentiment_score_finbert'] = sentiment_scores
    df['sentiment_positive_finbert'] = positive_scores
    df['sentiment_neutral_finbert'] = neutral_scores
    df['sentiment_negative_finbert'] = negative_scores


Device set to use mps:0


In [11]:
add_sentiment_scores(conbined_df, 'headline')
display(conbined_df.head())

Starting sentiment analysis...


Processing texts: 100%|██████████| 451165/451165 [2:06:32<00:00, 59.42text/s]  


,date,headline,sentiment_score_finbert,sentiment_positive_finbert,sentiment_neutral_finbert,sentiment_negative_finbert
0,2006-11-13 03:16:00-05:00,"Hitachi, GE boost alliance in nuclear power bu...",0.284925,0.826591,0.162399,0.011009
1,2006-11-13 08:45:00-05:00,"Volvo to cut 1,000 staff at Virginia plant",-0.348377,0.007027,0.025183,0.967790
2,2006-11-13 03:15:00-05:00,European banks hiding full pension obligations,0.003950,0.039472,0.938750,0.021779
3,2006-11-13 07:13:00-05:00,"Hitachi, GE to form joint nuclear power ventures",0.010726,0.059743,0.928143,0.012114
4,2006-11-13 07:29:00-05:00,Eddie Bauer agrees to be bought for $286 million,0.035885,0.156745,0.834435,0.008820


In [13]:
conbined_df.to_csv('finbert_sentiment_inserted.csv', index=False)

## LLM: Sentiment_score,relevance_score,event_importance,event _type

For the implementation of LLM features, please refer to `insert_llm_features.py`. The feature insertion is not implemented directly in this notebook due to the substantial size of our dataset. Processing each row sequentially would be computationally intensive, as each prompt request requires approximately 1.5 seconds on average, including data reading, prompt construction, and response generation. Given our dataset contains approximately 440,000 rows, sequential processing would require several days to complete.

The `insert_llm_features.py` script implements an asynchronous and batched prompting approach, achieving a processing rate of 50 iterations per second. This represents a significant performance improvement compared to sequential processing. However, due to the complexity of this implementation, it is more appropriate to execute it as a standalone Python script rather than within a Jupyter notebook environment.